In [ ]:
try:
    from google.colab import files
    uploaded = files.upload() 
    pdf_path = list(uploaded.keys())[0]
    print("Using PDF:", pdf_path)

except ImportError:
    pdf_path = "data/benchmark/flexqube_arsredovisning_2022.pdf" 
    
    print(f"Running locally, using: {pdf_path}")

Saving 68 sidor.pdf to 68 sidor.pdf


In [ ]:
!pip install PyMuPDF
import fitz

doc = fitz.open("/content/68 sidor.pdf")
text = ""

for i, page in enumerate(doc, start=1):
    page_text = page.get_text()
    text += f"--- PAGE {i} ---\n{page_text}\n\n"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 49.2 MB/s eta 0:00:00


In [ ]:
import requests
import json

headers = {
    'Authorization': 'API'
}

url = 'https://api.va.eu-west-1.landing.ai/v1/ade/parse/jobs'

# Upload a document
document = open('/content/68 sidor.pdf', 'rb')
files = {'document': document}
data = {'model': 'dpt-2-latest'}

response = requests.post(url, files=files, data=data, headers=headers)
print(response.json())

{'error': 'Authorization header should start with "Basic " or "Bearer "'}


In [ ]:
import requests

headers = {
    'Authorization': 'API'
}
job_id = "secret_id"
url = f"https://api.va.eu-west-1.landing.ai/v1/ade/parse/jobs/{'secret_id'}"

response = requests.get(url, headers=headers)
response_data = response.json()

print(response_data)

if response_data.get('status') == 'completed':
    data = response_data.get('data')
    if data and data.get('markdown'):
        markdown_content = data['markdown']
        with open('markdown-mri-report.md', 'w', encoding='utf-8') as f:
            f.write(markdown_content)
        print("Markdown content saved to a Markdown file.")
    elif response_data.get('output_url'):
        output_url = response_data['output_url']
        print("Downloading from output_url:", output_url)

        # Hämta filen UTAN Authorization-header
file_response = requests.get(output_url)
if file_response.status_code == 200:
    with open("landingai_output.md", "wb") as f:
        f.write(file_response.content)
    print("Markdown file downloaded and saved as landingai_output.md")
else:
    print("Failed to download file from output_url:", file_response.status_code, file_response.text)

{'error': 'Authorization header should start with "Basic " or "Bearer "'}
Markdown file downloaded and saved as landingai_output.md


In [ ]:
from google.colab import files
files.download("landingai_output.md")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from pathlib import Path
import re

# Läs markdownfilen
md = Path("landingai_output.md").read_text(encoding="utf-8")

# Splitta på LandingAI:s sidbrytare
pages = re.split(r"<!--\s*PAGE BREAK\s*-->", md)

new_md = []

for i, page in enumerate(pages, start=1):
    page = re.sub(r'<a id="page-\d+"></a>', "", page).strip()

    # Lägg till sidrubrik
    new_md.append(f"# Page {i}\n\n{page}")

# Lägg in --- mellan varje sida
final_md = "\n\n---\n\n".join(new_md)

# Spara ny fil
Path("landingai_output_paged.md").write_text(final_md, encoding="utf-8")

print(f"Klart! {len(pages)} sidor sparade i landingai_output_paged.md")

Klart! 68 sidor sparade i landingai_output_paged.md


In [ ]:
from google.colab import files
files.download("landingai_output_paged.md")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>